# Electricity Demand Seasonality Analysis

## Project Overview
Analyze electricity demand data to study hourly, daily, weekly, and seasonal usage patterns across different time scales.

## Learning Objectives
- Analyze multi-scale seasonality (hourly, daily, weekly, annual)
- Identify peak demand periods and load profiles
- Correlate demand with temperature and calendar effects
- Visualize load duration curves and demand heatmaps

## Problem Statement
Grid operators and energy planners need to understand demand patterns at multiple time scales to optimize generation scheduling, plan capacity, and manage peak loads.

## Why This Project Matters
Electricity cannot be stored economically at scale. Understanding demand seasonality is critical for grid reliability, cost management, and renewable energy integration.

## Dataset Overview
Synthetic hourly electricity demand: 2 years (~17,520 hours) with temperature, hour, day type, and seasonal patterns.

## Dataset Source and License Notes
- Synthetic data inspired by typical grid demand patterns
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n_hours = 365 * 2 * 24  # 2 years
dates = pd.date_range('2022-01-01', periods=n_hours, freq='h')
hour = dates.hour
month = dates.month
dow = dates.dayofweek  # 0=Mon

# Temperature: seasonal + diurnal
temp_seasonal = 15 + 15 * np.sin(2 * np.pi * (dates.dayofyear - 80) / 365.25)
temp_diurnal = 5 * np.sin(2 * np.pi * (hour - 6) / 24)
temperature = (temp_seasonal + temp_diurnal + np.random.normal(0, 3, n_hours)).round(1)

# Demand components
base_load = 3000
hourly_profile = np.array([0.7,0.65,0.62,0.60,0.62,0.68,0.80,0.95,1.05,1.10,1.12,1.10,
                            1.08,1.05,1.03,1.05,1.10,1.15,1.12,1.05,0.95,0.88,0.80,0.75])
hour_factor = np.array([hourly_profile[h] for h in hour])
# Weekend reduction
weekend_factor = np.where(dow >= 5, 0.85, 1.0)
# Seasonal: higher in summer (AC) and winter (heating)
seasonal_factor = 1.0 + 0.15 * np.abs(np.sin(2 * np.pi * (dates.dayofyear - 80) / 365.25))
# Temperature effect: U-shaped (heating below 10C, cooling above 25C)
temp_effect = np.where(temperature < 10, (10 - temperature) * 20,
              np.where(temperature > 25, (temperature - 25) * 25, 0))

demand = (base_load * hour_factor * weekend_factor * seasonal_factor + temp_effect
          + np.random.normal(0, 100, n_hours))
demand = np.clip(demand, 1000, 6000).round(0)

df = pd.DataFrame({'Datetime': dates, 'Demand_MW': demand, 'Temperature_C': temperature})
df = df.set_index('Datetime')
df['Hour'] = df.index.hour
df['DayOfWeek'] = df.index.day_name()
df['Month'] = df.index.month
df['IsWeekend'] = df.index.dayofweek >= 5
df['Date'] = df.index.date

print(f'Dataset: {df.shape}')
print(f'Demand range: {df["Demand_MW"].min():,.0f} — {df["Demand_MW"].max():,.0f} MW')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Hours: {len(df):,}')
print(f'\nDemand stats:\n{df["Demand_MW"].describe().round(0)}')

## Daily Demand Profile

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
weekday = df[~df['IsWeekend']].groupby('Hour')['Demand_MW'].mean()
weekend = df[df['IsWeekend']].groupby('Hour')['Demand_MW'].mean()
ax.plot(weekday.index, weekday.values, marker='o', label='Weekday', color='steelblue')
ax.plot(weekend.index, weekend.values, marker='s', label='Weekend', color='coral')
ax.set_title('Average Hourly Demand Profile')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Demand (MW)')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'hourly_profile.png'), dpi=100, bbox_inches='tight')
plt.show()

## Weekly Pattern

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
df.groupby('DayOfWeek')['Demand_MW'].mean().reindex(dow_order).plot.bar(ax=ax, edgecolor='black', color='teal')
ax.set_title('Average Demand by Day of Week')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'weekly_pattern.png'), dpi=100, bbox_inches='tight')
plt.show()

## Monthly Seasonality

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
monthly = df.groupby('Month')['Demand_MW'].agg(['mean', 'max', 'min'])
ax.bar(monthly.index, monthly['mean'], edgecolor='black', color='steelblue', alpha=0.7, label='Mean')
ax.plot(monthly.index, monthly['max'], marker='^', color='red', label='Max')
ax.plot(monthly.index, monthly['min'], marker='v', color='blue', label='Min')
ax.set_title('Monthly Demand Statistics')
ax.set_xlabel('Month')
ax.set_ylabel('Demand (MW)')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'monthly_demand.png'), dpi=100, bbox_inches='tight')
plt.show()

## Demand Heatmap (Hour × Month)

In [ ]:
pivot = df.pivot_table(values='Demand_MW', index='Hour', columns='Month', aggfunc='mean')
fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(pivot, cmap='YlOrRd', annot=True, fmt='.0f', ax=ax)
ax.set_title('Average Demand Heatmap: Hour × Month')
ax.set_ylabel('Hour of Day')
ax.set_xlabel('Month')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'demand_heatmap.png'), dpi=100, bbox_inches='tight')
plt.show()

## Temperature vs Demand

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(df['Temperature_C'], df['Demand_MW'], alpha=0.05, s=3)
# Binned average
temp_bins = pd.cut(df['Temperature_C'], bins=20)
binned = df.groupby(temp_bins, observed=True)['Demand_MW'].mean()
bin_centers = [interval.mid for interval in binned.index]
ax.plot(bin_centers, binned.values, 'r-o', linewidth=2, markersize=5, label='Binned mean')
ax.set_title('Temperature vs Demand (U-shaped relationship)')
ax.set_xlabel('Temperature (°C)')
ax.set_ylabel('Demand (MW)')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'temp_vs_demand.png'), dpi=100, bbox_inches='tight')
plt.show()

## Load Duration Curve

In [ ]:
sorted_demand = df['Demand_MW'].sort_values(ascending=False).values
hours_pct = np.arange(1, len(sorted_demand) + 1) / len(sorted_demand) * 100
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(hours_pct, sorted_demand, color='steelblue')
ax.fill_between(hours_pct, sorted_demand, alpha=0.2)
ax.set_title('Load Duration Curve')
ax.set_xlabel('% of Time')
ax.set_ylabel('Demand (MW)')
ax.axhline(sorted_demand[int(len(sorted_demand)*0.05)], color='red', linestyle='--', alpha=0.5,
           label=f'95th pct: {sorted_demand[int(len(sorted_demand)*0.05)]:,.0f} MW')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'load_duration.png'), dpi=100, bbox_inches='tight')
plt.show()

## Interpretation and Business Insight
- **Hourly profile** shows classic double-peak pattern (morning ramp + evening peak) on weekdays
- **Weekend demand** is ~15% lower — industrial/commercial load drops
- **Monthly seasonality** shows summer and winter peaks — the U-shaped temperature relationship confirms heating and cooling loads
- **Demand heatmap** reveals the highest stress: summer afternoons (hours 10-17, months 6-8)
- **Load duration curve** shows peak demand occurs < 5% of the time — expensive peaker plants run very few hours
- **Temperature** is the strongest weather driver — every degree above 25°C adds ~25 MW

## Limitations
- Synthetic data with simplified demand model
- No actual generation or pricing data
- No industrial/residential breakdown
- No renewable generation (solar/wind) data
- No special events or outage data

## How to Improve This Project
- Add renewable generation profiles (solar, wind)
- Include electricity pricing for price-demand analysis
- Build short-term load forecasting models
- Add customer segment breakdowns
- Include grid reliability metrics (SAIDI, SAIFI)

## Production Considerations
- Real-time demand monitoring dashboards
- Day-ahead and week-ahead load forecasting
- Peak demand alerts and demand response triggers
- Integration with weather forecasts for predictive scheduling

## Common Mistakes
- Forecasting demand without separate weekday/weekend models
- Ignoring the non-linear temperature-demand relationship
- Using calendar month averages that mask within-month variation
- Not accounting for holidays that behave like weekends

## Mini Challenge / Exercises
1. What hour has the largest weekday-weekend demand gap?
2. Calculate the peak-to-trough ratio for each month.
3. What temperature range has the lowest average demand?
4. If peak demand grows by 3% per year, estimate the new peak load in 5 years.

## Final Summary / Key Takeaways
- Electricity demand has strong multi-scale seasonality: hourly, daily, weekly, and annual
- Temperature is the dominant weather driver with a U-shaped effect
- Peak demand defines infrastructure costs but occurs infrequently
- Weekday/weekend distinction is essential for accurate forecasting
- Understanding load profiles enables smarter grid operation and investment planning